# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atulpatel-net/FlyRank_ML_In/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# ============================================================
# Setup: Connect to the FlyRank warehouse
# ============================================================

%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# Hugging Face token
HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN is None:
    HF_TOKEN = getpass.getpass("Enter your Hugging Face READ token: ")

# Connect DuckDB
con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

# Warehouse location
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients/*.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content/*.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d/*.parquet')"
}

print("✅ Connected successfully.")


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
✅ Connected successfully.


In [2]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:>12,} rows")

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule (Plain Words)

My baseline rule prioritizes pages that receive high search visibility (high impressions) but have a relatively low click-through rate (CTR) for their search position. These pages already appear in search results but attract fewer clicks than expected, making them good candidates for optimization through improved titles, meta descriptions, or content updates.

Reason Codes

LOW_CTR  -  The page has high visibility but a lower-than-expected CTR for its search position.The page has high visibility but a lower-than-expected CTR for its search position.

HIGH_IMPRESSIONS  -   The page receives high search visibility and is a high-impact optimization opportunity.

LOW_VISIBILITY    -   The page has limited impressions and is a lower-priority opportunity.

REVIEW  	-      The page does not clearly match another rule and should be reviewed manually.

In [3]:
#Signal 1 CTR vs Position

signal1 = con.sql(f"""
WITH base AS (
    SELECT
        gsc_avg_position,
        gsc_clicks,
        gsc_impressions,
        CASE
            WHEN gsc_impressions > 0
            THEN 100.0 * gsc_clicks / gsc_impressions
            ELSE NULL
        END AS ctr
    FROM {TABLES['fact_daily_sample']}
    WHERE
        gsc_data_available IS TRUE
        AND gsc_impressions >= 20
)

SELECT
    CASE
        WHEN gsc_avg_position <= 3 THEN 'Top 3'
        WHEN gsc_avg_position <= 10 THEN 'Top 10'
        WHEN gsc_avg_position <= 20 THEN 'Top 20'
        ELSE '20+'
    END AS position_bucket,

    COUNT(*) AS n,
    ROUND(AVG(ctr),2) AS avg_ctr

FROM base

GROUP BY position_bucket

ORDER BY
CASE position_bucket
    WHEN 'Top 3' THEN 1
    WHEN 'Top 10' THEN 2
    WHEN 'Top 20' THEN 3
    ELSE 4
END
""").df()

display(signal1)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,avg_ctr
0,Top 3,77135,0.64
1,Top 10,796524,0.48
2,Top 20,275114,0.44
3,20+,231744,0.27


In [4]:
# Signal 2 Impressions / Volume

signal2 = con.sql(f"""
WITH base AS (
    SELECT
        gsc_impressions
    FROM {TABLES['fact_daily_sample']}
    WHERE
        gsc_data_available IS TRUE
        AND gsc_impressions > 0
)

SELECT
    CASE
        WHEN gsc_impressions >= 1000 THEN 'High (1000+)'
        WHEN gsc_impressions >= 500 THEN 'Medium (500-999)'
        WHEN gsc_impressions >= 100 THEN 'Low (100-499)'
        ELSE 'Very Low (<100)'
    END AS impression_bucket,

    COUNT(*) AS n,
    ROUND(AVG(gsc_impressions), 2) AS avg_impressions

FROM base

GROUP BY impression_bucket

ORDER BY
CASE impression_bucket
    WHEN 'High (1000+)' THEN 1
    WHEN 'Medium (500-999)' THEN 2
    WHEN 'Low (100-499)' THEN 3
    ELSE 4
END
""").df()

display(signal2)

,impression_bucket,n,avg_impressions
0,High (1000+),25519,1977.93
1,Medium (500-999),48600,685.29
2,Low (100-499),360041,208.83
3,Very Low (<100),3444777,16.61


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:

import os

os.makedirs("work/outputs", exist_ok=True)

con.sql(f"""
COPY (

WITH page_metrics AS (

    SELECT

        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions)                  AS impressions,
        SUM(gsc_clicks)                       AS clicks,
        AVG(gsc_avg_position)                 AS avg_position,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
            ELSE 0
        END                                   AS ctr

    FROM {TABLES['fact_daily_sample']}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
),

baseline AS (

SELECT

    *,

    ----------------------------------------------------------
    -- Weighted Baseline Score
    ----------------------------------------------------------

    (

        --------------------------------------------------
        -- Traffic Importance (40 pts)
        --------------------------------------------------

        CASE
            WHEN impressions >= 500000 THEN 40
            WHEN impressions >= 250000 THEN 30
            WHEN impressions >= 100000 THEN 20
            WHEN impressions >= 50000  THEN 10
            ELSE 0
        END

        +

        --------------------------------------------------
        -- CTR Opportunity (40 pts)
        --------------------------------------------------

        CASE
            WHEN ctr < 0.10 THEN 40
            WHEN ctr < 0.30 THEN 30
            WHEN ctr < 0.60 THEN 20
            WHEN ctr < 1.00 THEN 10
            ELSE 0
        END

        +

        --------------------------------------------------
        -- Ranking Opportunity (20 pts)
        --------------------------------------------------

        CASE
            WHEN avg_position <= 3 THEN 20
            WHEN avg_position <= 10 THEN 10
            ELSE 0
        END

    ) AS score,

    ----------------------------------------------------------
    -- Reason Code
    ----------------------------------------------------------

    CASE

        WHEN impressions >= 500000
             AND ctr < 0.30
            THEN 'URGENT_CTR_FIX'

        WHEN impressions >= 100000
             AND ctr < 0.60
            THEN 'LOW_CTR'

        WHEN impressions >= 250000
            THEN 'HIGH_TRAFFIC'

        WHEN avg_position <= 10
            THEN 'GOOD_POSITION'

        ELSE 'LOW_PRIORITY'

    END AS reason_code,

    ----------------------------------------------------------
    -- Action
    ----------------------------------------------------------

    CASE

        WHEN impressions >= 500000
             AND ctr < 0.30
            THEN 'Improve title and meta description immediately'

        WHEN impressions >= 100000
             AND ctr < 0.60
            THEN 'Review CTR and optimize snippet'

        WHEN impressions >= 250000
            THEN 'Monitor high-traffic page'

        WHEN avg_position <= 10
            THEN 'Track performance'

        ELSE 'No immediate action'

    END AS action

FROM page_metrics

)

SELECT *

FROM baseline

ORDER BY

    score DESC,
    impressions DESC,
    ctr ASC,
    avg_position ASC

)

TO 'work/outputs/baseline_action_score.csv'

WITH (
    FORMAT CSV,
    HEADER TRUE
)
""")

print("✅ Ranked queue created successfully.")



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Ranked queue created successfully.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [16]:
top20 = con.sql("""
SELECT *
FROM read_csv_auto('work/outputs/baseline_action_score.csv')
LIMIT 20
""").df()

display(top20)

,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr,score,reason_code,action
0,client_e547b89c05043229,content_963de14b1f58978f,615012,1676,6.397556,0.272515,80,URGENT_CTR_FIX,Improve title and meta description immediately
1,client_e547b89c05043229,content_545bb6cc7081ded3,585712,3048,2.110776,0.520392,80,LOW_CTR,Review CTR and optimize snippet
2,client_8ddc46da5414ffd8,content_943dc881428182b8,292416,399,2.689206,0.136449,80,LOW_CTR,Review CTR and optimize snippet
3,client_e547b89c05043229,content_9ef3d7516483e665,269943,787,2.098715,0.291543,80,LOW_CTR,Review CTR and optimize snippet
4,client_e547b89c05043229,content_eadb33b5df496f4a,591696,3817,2.258612,0.645095,70,HIGH_TRAFFIC,Monitor high-traffic page
5,client_a80fca3f171ed1de,content_c1f764a2f362d1c3,235427,45,7.132457,0.019114,70,LOW_CTR,Review CTR and optimize snippet
6,client_a80fca3f171ed1de,content_9540d884af3e41fd,171004,51,8.757071,0.029824,70,LOW_CTR,Review CTR and optimize snippet
7,client_73cda7b4e4f265ea,content_33d31496fca9665e,169233,23,7.713277,0.013591,70,LOW_CTR,Review CTR and optimize snippet
8,client_73cda7b4e4f265ea,content_f43118e089ecc69a,145592,107,7.672917,0.073493,70,LOW_CTR,Review CTR and optimize snippet
9,client_8ddc46da5414ffd8,content_d0acf7062bc6b257,145022,46,3.049897,0.031719,70,LOW_CTR,Review CTR and optimize snippet


### Top-20 Review

| Rank | Action | Reason Code | Confidence | What would make it wrong? |
|------|--------|-------------|------------|----------------------------|
| 1 | Improve title and meta description immediately | URGENT_CTR_FIX | Very High | The page may naturally receive low CTR because of informational search intent or SERP features. |
| 2 | Review CTR and optimize snippet | LOW_CTR | High | CTR may improve naturally if search intent or rankings change over time. |
| 3 | Review CTR and optimize snippet | LOW_CTR | High | Low CTR could be caused by featured snippets or competitor-rich results rather than poor metadata. |
| 4 | Review CTR and optimize snippet | LOW_CTR | High | Recent content updates may not yet be reflected in search performance. |
| 5 | Monitor high-traffic page | HIGH_TRAFFIC | High | The page already attracts many clicks, so further optimization may have limited impact. |
| 6 | Review CTR and optimize snippet | LOW_CTR | High | Very low CTR may result from broad or irrelevant search queries instead of poor titles. |
| 7 | Review CTR and optimize snippet | LOW_CTR | High | Ranking fluctuations or seasonal trends could temporarily reduce CTR. |
| 8 | Review CTR and optimize snippet | LOW_CTR | High | Search demand may have changed since the data was collected. |
| 9 | Review CTR and optimize snippet | LOW_CTR | High | Some impressions may come from low-intent queries that rarely generate clicks. |
| 10 | Review CTR and optimize snippet | LOW_CTR | High | SERP features such as AI Overviews or featured snippets may reduce organic CTR. |
| 11 | Review CTR and optimize snippet | LOW_CTR | High | External events or seasonal traffic could explain the low CTR. |
| 12 | Review CTR and optimize snippet | LOW_CTR | High | The page title may already have been updated after this data snapshot. |
| 13 | Review CTR and optimize snippet | LOW_CTR | High | High impressions do not always indicate strong commercial value. |
| 14 | Review CTR and optimize snippet | LOW_CTR | High | The page could already be under optimization, making historical data less representative. |
| 15 | Review CTR and optimize snippet | LOW_CTR | High | Query intent may not align well with the page content. |
| 16 | Review CTR and optimize snippet | LOW_CTR | High | Performance differences across devices or locations may affect CTR. |
| 17 | Track performance | GOOD_POSITION | Medium | The page already ranks well; improving CTR may require richer snippets rather than content changes. |
| 18 | Track performance | GOOD_POSITION | Medium | Good rankings do not always translate into higher clicks if user intent is weak. |
| 19 | Review CTR and optimize snippet | LOW_CTR | High | Although impressions are high, CTR may be limited by competitive search results. |
| 20 | Review CTR and optimize snippet | LOW_CTR | High | Additional metrics such as page freshness or content quality may be needed before taking action. |

### Summary

The baseline prioritizes pages with high search visibility but relatively low click-through rates, as these pages offer the greatest opportunity for improving organic traffic. The review also shows that every recommendation should be validated manually because factors such as search intent, SERP features, seasonality, and recent content updates can influence CTR without indicating a true optimization problem.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Although the baseline successfully prioritizes high-impact pages, some recommendations may be weak:

- Pages with high impressions but already reasonable CTR may not benefit significantly from title or meta description changes.
- Pages with good average positions can still have low CTR because of SERP features, branded searches, or informational search intent rather than poor optimization.
- The baseline does not consider content freshness, page quality, or user intent, so some recommendations may require manual review before action.

## Leakage Check

No product flags or future-window information were used while building the baseline.

The score was computed only from features available at the decision time:

- Total Google Search Console impressions
- Total Google Search Console clicks
- Average search position
- Calculated click-through rate (CTR)

The baseline does **not** use:

- Any future performance metrics
- Any label-derived columns
- Any FlyRank product flags or manually assigned actions

Therefore, the ranked queue represents an honest rule-based baseline that can be fairly compared with future machine learning models.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.